# Install Dependencies for BiLSTM-CRF

In [1]:
!pip install pytorch-crf seqeval tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=60a0e6daeb90787fde55a2052e2f637e1dbe094002f70afd294b3bcd6e31864e
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


# 1. Split dataset into 70 20 10 + balanced

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from IPython.display import display

# reproducibility
np.random.seed(42)

# load data
df = pd.concat([
    pd.read_csv("train_clean.csv"),
    pd.read_csv("test_clean.csv"),
    pd.read_csv("validation_clean.csv")
], ignore_index=True)

# build document-level label
doc_labels = df.groupby("document_id")["entity_type"].unique().reset_index()

# convert label list → string (for stratification)
doc_labels["label"] = doc_labels["entity_type"].apply(lambda x: "_".join(sorted(x)))

# split doc_ids
train_ids, temp_ids = train_test_split(
    doc_labels["document_id"],
    test_size=0.3,
    stratify=doc_labels["label"],
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=1/3,  # 0.1 total
    stratify=doc_labels.set_index("document_id").loc[temp_ids]["label"],
    random_state=42
)

# build final datasets
train_df = df[df["document_id"].isin(train_ids)]
val_df = df[df["document_id"].isin(val_ids)]
test_df = df[df["document_id"].isin(test_ids)]

# --- PRO TABLE OUTPUT ---
summary_df = pd.DataFrame({
    "Train Size": [f"{len(train_df):,}"],      # Added commas for thousand separators
    "Validation Size": [f"{len(val_df):,}"],
    "Test Size": [f"{len(test_df):,}"]
})

# Apply custom styles (hiding index, adding a title, and styling headers)
pro_table = (
    summary_df.style
    .hide(axis="index")
    .set_caption("Dataset Split Summary")
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('margin-bottom', '10px')]},
        {'selector': 'th', 'props': [('background-color', '#4CAF50'), ('color', 'white'), ('text-align', 'center'), ('padding', '10px')]},
        {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '10px'), ('font-size', '14px')]}
    ])
)

display(pro_table)

Train Size,Validation Size,Test Size
"20,243","6,171","3,010"


# 2. DATA CLEANING



In [3]:
import pandas as pd
from IPython.display import display, Markdown

# 1. Initialize datasets from the previous split
train = train_df.copy()
val = val_df.copy()
test = test_df.copy()

# ==========================================
# TABLE 1: Column Info & Missing Values
# ==========================================
display(Markdown("## Column Info & Missing Values (All Splits)"))

info_data = []
for col in train.columns:
    info_data.append({
        "Feature": col,
        "Data Type": str(train[col].dtype),
        "Train Missing": train[col].isnull().sum(),
        "Val Missing": val[col].isnull().sum(),
        "Test Missing": test[col].isnull().sum()
    })

info_df = pd.DataFrame(info_data)
display(info_df.style.hide(axis="index").set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#4CAF50'), ('color', 'white'), ('text-align', 'center'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '8px')]}
]))

# ==========================================
# DATA CLEANING
# ==========================================
display(Markdown("## Validation Data Cleaning"))
val_before = len(val)
val = val.dropna(subset=["entity_text"])
val_after = len(val)

print(f"Dropped {val_before - val_after} rows with missing 'entity_text' from Validation.")
print(f"Validation size after cleaning: {val_after:,}\n")

# ==========================================
# TABLE 2: Entity Type Distribution
# ==========================================
display(Markdown("## Entity Type Distribution"))

dist_df = pd.DataFrame({
    "Entity Type": ["Chemical", "Disease"],
    "Train": [
        f"{(train['entity_type'] == 'Chemical').sum():,}",
        f"{(train['entity_type'] == 'Disease').sum():,}"
    ],
    "Validation": [
        f"{(val['entity_type'] == 'Chemical').sum():,}",
        f"{(val['entity_type'] == 'Disease').sum():,}"
    ],
    "Test": [
        f"{(test['entity_type'] == 'Chemical').sum():,}",
        f"{(test['entity_type'] == 'Disease').sum():,}"
    ]
})

display(dist_df.style.hide(axis="index").set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2196F3'), ('color', 'white'), ('text-align', 'center'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '8px')]}
]))

# ==========================================
# DATA PREVIEW
# ==========================================
display(Markdown("**Data Preview (First 5 Rows & 5 Columns):**"))
preview_cols = ["document_id", "passage_type", "entity_text", "entity_type", "text"]
preview_df = train[preview_cols].head(5).copy()

# Shorten the 'text' column so it doesn't stretch the screen
preview_df["text"] = preview_df["text"].apply(lambda x: x[:75] + "..." if len(str(x)) > 75 else x)

display(preview_df)

## Column Info & Missing Values (All Splits)

Feature,Data Type,Train Missing,Val Missing,Test Missing
document_id,int64,0,0,0
passage_type,object,0,0,0
text,object,0,0,0
entity_text,object,0,7,0
entity_type,object,0,0,0
start,int64,0,0,0
end,int64,0,0,0


## Validation Data Cleaning

Dropped 7 rows with missing 'entity_text' from Validation.
Validation size after cleaning: 6,164



## Entity Type Distribution

Entity Type,Train,Validation,Test
Chemical,"10,969","3,392","1,595"
Disease,"9,274","2,772","1,415"


**Data Preview (First 5 Rows & 5 Columns):**

,document_id,passage_type,entity_text,entity_type,text
0,227508,title,Naloxone,Chemical,Naloxone reverses the antihypertensive effect ...
1,227508,title,clonidine,Chemical,Naloxone reverses the antihypertensive effect ...
2,227508,abstract,hypertensive,Disease,"In unanesthetized, spontaneously hypertensive ..."
3,227508,abstract,clonidine,Chemical,"In unanesthetized, spontaneously hypertensive ..."
4,227508,abstract,nalozone,Chemical,"In unanesthetized, spontaneously hypertensive ..."


# 3. GROUP ENTITIES

In [4]:
def group_entities_fixed(df):
    grouped = df.groupby("text")
    data = []
    for text, group in grouped:
        entities = []
        for _, row in group.iterrows():
            entity_text = row["entity_text"]
            start = text.find(entity_text)
            if start == -1:
                continue
            end = start + len(entity_text)
            entities.append({
                "start": start,
                "end": end,
                "label": row["entity_type"]
            })
        data.append({
            "text": text,
            "entities": entities
        })
    return data

train_grouped = group_entities_fixed(train)
val_grouped = group_entities_fixed(val)
test_grouped = group_entities_fixed(test)

In [5]:
import json

def show_examples_clean(grouped_data, num_examples=1):
    for i in range(num_examples):
        example = grouped_data[i]

        # Build a cleaner list of entities with the actual text included
        clean_entities = []
        for ent in example["entities"]:
            clean_entities.append({
                "extracted_text": example["text"][ent["start"]:ent["end"]],
                "label": ent["label"],
                "start": ent["start"],
                "end": ent["end"]
            })

        # Structure the final output dictionary
        clean_output = {
            "text": example["text"],
            "entities": clean_entities
        }

        print(f"========== Example ==========")
        # Print with an indent of 4 spaces for that clean, structured look
        print(json.dumps(clean_output, indent=4))
        print("\n")

show_examples_clean(train_grouped, 1)

========== Example ==========
{
    "text": "-Tacrolimus (FK 506) is a powerful, widely used immunosuppressant. The clinical utility of FK 506 is complicated by substantial hypertension and nephrotoxicity. To clarify the mechanisms of FK 506-induced hypertension, we studied the chronic effects of FK 506 on the synthesis of endothelin-1 (ET-1), the expression of mRNA of ET-1 and endothelin-converting enzyme-1 (ECE-1), the endothelial nitric oxide synthase (eNOS) activity, and the expression of mRNA of eNOS and C-type natriuretic peptide (CNP) in rat blood vessels. In addition, the effect of the specific endothelin type A receptor antagonist FR 139317 on FK 506-induced hypertension in rats was studied. FK 506, 5 mg. kg-1. d-1 given for 4 weeks, elevated blood pressure from 102+/-13 to 152+/-15 mm Hg and increased the synthesis of ET-1 and the levels of ET-1 mRNA in the mesenteric artery (240% and 230%, respectively). Little change was observed in the expression of ECE-1 mRNA and CNP mRNA

# 4. CONVERT TO BIO FORMAT

In [21]:
import re
import pandas as pd
from IPython.display import display, Markdown

def text_to_bio(text, entities):
    tokens = []
    labels = []
    for match in re.finditer(r'\w+|[^\w\s]', text):
        token = match.group()
        start = match.start()
        end = match.end()
        label = "O"
        for ent in entities:
            ent_start = ent["start"]
            ent_end = ent["end"]
            ent_label = ent["label"]
            if not (end <= ent_start or start >= ent_end):
                if re.match(r'\w+', token):
                    if start == ent_start:
                        label = "B-" + ent_label
                    else:
                        label = "I-" + ent_label
        tokens.append(token)
        labels.append(label)
    return tokens, labels

def convert_dataset(grouped_data):
    dataset = []
    for item in grouped_data:
        tokens, labels = text_to_bio(item["text"], item["entities"])
        dataset.append({
            "tokens": tokens,
            "labels": labels
        })
    return dataset

# 1. Convert all datasets
train_bio = convert_dataset(train_grouped)
val_bio = convert_dataset(val_grouped)
test_bio = convert_dataset(test_grouped)

# ==========================================
# SHOW EXAMPLE OF BIO CONVERSION (VECTORS)
# ==========================================
display(Markdown("##BIO Format Example (Vectors)"))

example = train_bio[0]

# Display as raw Python lists (vectors)
print("TOKENS VECTOR:")
print(example["tokens"][:15], "...\n")

print("LABELS VECTOR:")
print(example["labels"][:15], "...\n")

# Display as a horizontal 1xN Table
display(Markdown("**Horizontal Table View (First 15 tokens):**"))

# Create a DataFrame where columns are tokens and the single row is the labels
horizontal_df = pd.DataFrame([example["labels"][:15]], columns=example["tokens"][:15], index=["BIO Tag"])

# FIXED STYLING: Added ('color', 'black') to 'th' so it pops in Dark Mode
display(horizontal_df.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#f4f4f4'), ('color', 'black'), ('border', '1px solid #ddd'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('text-align', 'center'), ('padding', '8px')]}
]))

##BIO Format Example (Vectors)

TOKENS VECTOR:
['-', 'Tacrolimus', '(', 'FK', '506', ')', 'is', 'a', 'powerful', ',', 'widely', 'used', 'immunosuppressant', '.', 'The'] ...

LABELS VECTOR:
['O', 'B-Chemical', 'O', 'B-Chemical', 'I-Chemical', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'] ...



**Horizontal Table View (First 15 tokens):**

,-,Tacrolimus,(,FK,506,),is,a,powerful,",",widely,used,immunosuppressant,.,The
BIO Tag,O,B-Chemical,O,B-Chemical,I-Chemical,O,O,O,O,O,O,O,O,O,O


# BIO to CSV

In [7]:
import pandas as pd

# ==========================================
# EXPORT BIO FORMAT TO CSV (2-Row Layout)
# ==========================================
print("Exporting BIO data to 2-row horizontal CSVs...")

def export_2row_csv(bio_data, split_prefix, filename):
    rows = []

    for sentence_idx, item in enumerate(bio_data):
        sentence_id = f"{split_prefix}_{sentence_idx}"

        # Row 1: Tokens
        token_row = {"Sentence_ID": sentence_id, "Data_Type": "Tokens"}
        for i, token in enumerate(item["tokens"]):
            token_row[f"Pos_{i}"] = token

        # Row 2: Tags
        tag_row = {"Sentence_ID": sentence_id, "Data_Type": "BIO Tags"}
        for i, tag in enumerate(item["labels"]):
            tag_row[f"Pos_{i}"] = tag

        # Append the 2 rows + 1 empty spacer row for readability
        rows.append(token_row)
        rows.append(tag_row)
        rows.append({})

    # Create DataFrame and save
    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)
    return len(bio_data)

# Run the export
train_count = export_2row_csv(train_bio, "train", "train_bio_horizontal.csv")
val_count = export_2row_csv(val_bio, "val", "val_bio_horizontal.csv")
test_count = export_2row_csv(test_bio, "test", "test_bio_horizontal.csv")

print("Successfully saved horizontal BIO format to CSV!")
print(f"train_bio_horizontal.csv  | {train_count:,} texts")
print(f"val_bio_horizontal.csv    | {val_count:,} texts")
print(f"test_bio_horizontal.csv   | {test_count:,} texts")

Exporting BIO data to 2-row horizontal CSVs...
Successfully saved horizontal BIO format to CSV!
train_bio_horizontal.csv  | 2,076 texts
val_bio_horizontal.csv    | 594 texts
test_bio_horizontal.csv   | 297 texts


# 5. BUILD VOCABULARY & DATASETS FOR PYTORCH

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from IPython.display import display, Markdown

# Build word vocabulary from training data
words = set()
for item in train_bio:
    for token in item['tokens']:
        words.add(token.lower())

# Define Special Tokens
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

word2idx = {w: i + 2 for i, w in enumerate(words)}
word2idx[PAD_TOKEN] = 0
word2idx[UNK_TOKEN] = 1
idx2word = {i: w for w, i in word2idx.items()}

# Define Label Dictionary
tag2idx = {
    "<PAD>": 0,
    "O": 1,
    "B-Chemical": 2,
    "I-Chemical": 3,
    "B-Disease": 4,
    "I-Disease": 5
}
idx2tag = {i: t for t, i in tag2idx.items()}

# ==========================================
# SHOW VOCABULARY & TAGS SUMMARY
# ==========================================
display(Markdown("##Vocabulary & Tags Summary"))
display(Markdown(f"**Total Vocabulary Size:** `{len(word2idx):,}` unique tokens"))

# Convert the dictionary to a DataFrame for clean display
tags_df = pd.DataFrame(list(tag2idx.items()), columns=["Tag / Label", "Index ID"])

# FIX: Escape HTML characters so <PAD> doesn't turn invisible!
tags_df["Tag / Label"] = tags_df["Tag / Label"].str.replace("<", "&lt;").str.replace(">", "&gt;")

# Apply styling (Orange header for variety)
display(tags_df.style.hide(axis="index").set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#FF9800'), ('color', 'white'), ('text-align', 'center'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '8px'), ('font-family', 'monospace')]}
]))

##Vocabulary & Tags Summary

**Total Vocabulary Size:** `12,414` unique tokens

Tag / Label,Index ID
<PAD>,0
O,1
B-Chemical,2
I-Chemical,3
B-Disease,4
I-Disease,5


In [9]:
class NERDataset(Dataset):
    def __init__(self, data, word2idx, tag2idx, max_len=256):
        self.data = data
        self.word2idx = word2idx
        self.tag2idx = tag2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item['tokens']
        tags = item['labels']

        # Convert to IDs
        word_ids = [self.word2idx.get(t.lower(), self.word2idx[UNK_TOKEN]) for t in tokens][:self.max_len]
        tag_ids = [self.tag2idx[t] for t in tags][:self.max_len]

        # Padding
        pad_len = self.max_len - len(word_ids)
        word_ids.extend([self.word2idx[PAD_TOKEN]] * pad_len)
        tag_ids.extend([self.tag2idx[PAD_TOKEN]] * pad_len)

        # Mask for CRF (1 for real tokens, 0 for padding)
        mask = [1] * min(len(tokens), self.max_len) + [0] * pad_len

        return torch.tensor(word_ids), torch.tensor(tag_ids), torch.tensor(mask, dtype=torch.bool)

MAX_LEN = 256
BATCH_SIZE = 16

train_dataset = NERDataset(train_bio, word2idx, tag2idx, max_len=MAX_LEN)
val_dataset = NERDataset(val_bio, word2idx, tag2idx, max_len=MAX_LEN)
test_dataset = NERDataset(test_bio, word2idx, tag2idx, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# 6. BILSTM-CRF MODEL DEFINITION

In [10]:
import torch.nn as nn
from torchcrf import CRF

class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=128, hidden_dim=256):
        super(BiLSTM_CRF, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # BiLSTM Layer
        self.lstm = nn.LSTM(embedding_dim, hidden_dim // 2,
                            num_layers=1, bidirectional=True, batch_first=True)

        # Maps the output of the LSTM into tag space
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

        # CRF Layer
        self.crf = CRF(tagset_size, batch_first=True)

    def forward(self, x, mask):
        # Get LSTM features
        embs = self.embedding(x)
        lstm_out, _ = self.lstm(embs)
        emissions = self.hidden2tag(lstm_out)
        return emissions

    def loss(self, x, tags, mask):
        emissions = self.forward(x, mask)
        # Calculate negative log likelihood loss
        return -self.crf(emissions, tags, mask=mask, reduction='mean')

    def decode(self, x, mask):
        emissions = self.forward(x, mask)
        # Find the best path using Viterbi algorithm
        return self.crf.decode(emissions, mask=mask)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTM_CRF(vocab_size=len(word2idx), tagset_size=len(tag2idx)).to(device)

# 7. TRAINING LOOP

In [11]:
import torch.optim as optim
from tqdm import tqdm

optimizer = optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 10  # More epochs needed for BiLSTM compared to BERT

print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for words_b, tags_b, mask_b in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        words_b, tags_b, mask_b = words_b.to(device), tags_b.to(device), mask_b.to(device)

        optimizer.zero_grad()
        loss = model.loss(words_b, tags_b, mask_b)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")

Starting Training...


Epoch 1/10: 100%|██████████| 130/130 [00:34<00:00,  3.73it/s]


Epoch 1 Loss: 42.2797


Epoch 2/10: 100%|██████████| 130/130 [00:33<00:00,  3.88it/s]


Epoch 2 Loss: 23.7321


Epoch 3/10: 100%|██████████| 130/130 [00:34<00:00,  3.74it/s]


... (11 outputs left out)

Epoch 9/10: 100%|██████████| 130/130 [00:33<00:00,  3.85it/s]


Epoch 9 Loss: 4.5777


Epoch 10/10: 100%|██████████| 130/130 [00:33<00:00,  3.91it/s]

Epoch 10 Loss: 3.4076


# 8. EVALUATION ON TEST SET

In [18]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

model.eval()

# --- 1. Test Set Predictions ---
true_labels = []
pred_labels = []

with torch.no_grad():
    for words_b, tags_b, mask_b in test_loader:
        words_b, tags_b, mask_b = words_b.to(device), tags_b.to(device), mask_b.to(device)

        # Predict
        predictions = model.decode(words_b, mask_b)

        # Extract true labels based on mask length
        tags_b_cpu = tags_b.cpu().numpy()
        mask_lengths = mask_b.sum(dim=1).cpu().numpy()

        for i in range(len(predictions)):
            seq_len = mask_lengths[i]

            # True tags for this sequence (ignoring <PAD>)
            true_seq = [idx2tag[tag] for tag in tags_b_cpu[i][:seq_len]]
            # Predicted tags for this sequence
            pred_seq = [idx2tag[tag] for tag in predictions[i]]

            true_labels.append(true_seq)
            pred_labels.append(pred_seq)

# --- 2. Validation Set Predictions ---
true_val_labels = []
pred_val_labels = []

with torch.no_grad():
    for words_b, tags_b, mask_b in val_loader:
        words_b, tags_b, mask_b = words_b.to(device), tags_b.to(device), mask_b.to(device)

        # Predict
        predictions = model.decode(words_b, mask_b)

        # Extract true labels based on mask length
        tags_b_cpu = tags_b.cpu().numpy()
        mask_lengths = mask_b.sum(dim=1).cpu().numpy()

        for i in range(len(predictions)):
            seq_len = mask_lengths[i]

            # True tags for this sequence (ignoring <PAD>)
            true_seq = [idx2tag[tag] for tag in tags_b_cpu[i][:seq_len]]
            # Predicted tags for this sequence
            pred_seq = [idx2tag[tag] for tag in predictions[i]]

            true_val_labels.append(true_seq)
            pred_val_labels.append(pred_seq)

# ==========================================
# OUTPUT FORMATTING (TABLES)
# ==========================================
display(Markdown("##Model Evaluation Metrics"))

# 1. Global Metrics Table
global_metrics = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1-Score", "Accuracy"],
    "Test Set": [
        precision_score(true_labels, pred_labels),
        recall_score(true_labels, pred_labels),
        f1_score(true_labels, pred_labels),
        accuracy_score(true_labels, pred_labels)
    ],
    "Validation Set": [
        precision_score(true_val_labels, pred_val_labels),
        recall_score(true_val_labels, pred_val_labels),
        f1_score(true_val_labels, pred_val_labels),
        accuracy_score(true_val_labels, pred_val_labels)
    ]
})

display(Markdown("###Global Overall Metrics"))
display(global_metrics.style.hide(axis="index").format({
    "Test Set": "{:.4f}",
    "Validation Set": "{:.4f}"
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2196F3'), ('color', 'white'), ('text-align', 'center'), ('padding', '10px')]},
    {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '10px')]}
]))

# 2. Detailed Classification Reports
# By setting output_dict=True, seqeval gives us a dictionary instead of a messy string
test_dict = classification_report(true_labels, pred_labels, output_dict=True)
val_dict = classification_report(true_val_labels, pred_val_labels, output_dict=True)

df_test = pd.DataFrame(test_dict).T
df_val = pd.DataFrame(val_dict).T

# Function to style our detailed reports consistently
def style_detailed_report(df, title, color):
    display(Markdown(f"### {title}"))
    styled = (df.style
        .format("{:.4f}", subset=["precision", "recall", "f1-score"])
        .format("{:.0f}", subset=["support"], na_rep="-")
        .set_table_styles([
            {'selector': 'th', 'props': [(f'background-color', color), ('color', 'white'), ('text-align', 'center'), ('padding', '8px')]},
            {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled)

# Display the two detailed reports with different colored headers to tell them apart
style_detailed_report(df_test, "Detailed Classification Report (Test Set)", "#4CAF50")
style_detailed_report(df_val, "Detailed Classification Report (Validation Set)", "#FF9800")

##Model Evaluation Metrics

###Global Overall Metrics

Metric,Test Set,Validation Set
Precision,0.6138,0.6094
Recall,0.5597,0.5633
F1-Score,0.5855,0.5854
Accuracy,0.9482,0.9502


### Detailed Classification Report (Test Set)

,precision,recall,f1-score,support
Chemical,0.5839,0.6399,0.6106,897
Disease,0.6594,0.4786,0.5546,886
micro avg,0.6138,0.5597,0.5855,1783
macro avg,0.6217,0.5592,0.5826,1783
weighted avg,0.6214,0.5597,0.5828,1783


### Detailed Classification Report (Validation Set)

,precision,recall,f1-score,support
Chemical,0.5674,0.6283,0.5963,1816
Disease,0.6774,0.4938,0.5712,1701
micro avg,0.6094,0.5633,0.5854,3517
macro avg,0.6224,0.5611,0.5838,3517
weighted avg,0.6206,0.5633,0.5842,3517


# 9. MANUAL TESTING

In [20]:
import re
import torch
import pandas as pd
from IPython.display import display, Markdown

def predict_sentence_styled(text):
    model.eval()

    # Tokenize the input exactly like training
    tokens = []
    for match in re.finditer(r'\w+|[^\w\s]', text):
        tokens.append(match.group())

    word_ids = [word2idx.get(t.lower(), word2idx[UNK_TOKEN]) for t in tokens]

    words_tensor = torch.tensor([word_ids]).to(device)
    mask_tensor = torch.tensor([[1] * len(word_ids)], dtype=torch.bool).to(device)

    with torch.no_grad():
        predictions = model.decode(words_tensor, mask_tensor)[0]

    labels = [idx2tag[pred_idx] for pred_idx in predictions]

    # Display the original text
    display(Markdown(f"**Input:** *{text}*"))

    # Build the horizontal wrapped grid (12 words per line)
    chunk_size = 12
    grid_rows = []

    for i in range(0, len(tokens), chunk_size):
        t_chunk = tokens[i : i + chunk_size]
        l_chunk = labels[i : i + chunk_size]

        # Pad the chunks with empty strings to keep the table perfectly aligned
        t_chunk += [""] * (chunk_size - len(t_chunk))
        l_chunk += [""] * (chunk_size - len(l_chunk))

        grid_rows.append(t_chunk)
        grid_rows.append(l_chunk)

    grid_df = pd.DataFrame(grid_rows)

    # Custom styler to make the predictions pop visually
    def style_pred_grid(x):
        df_style = pd.DataFrame('', index=x.index, columns=x.columns)
        for r in range(len(x)):
            for c in range(len(x.columns)):
                val = str(x.iloc[r, c])
                if r % 2 == 1:  # This is a Label Row
                    if val.startswith('B-') or val.startswith('I-'):
                        # Highlight entities in a nice purple box
                        df_style.iloc[r, c] = 'background-color: #E1BEE7; color: black; font-weight: bold; text-align: center; border-bottom: 3px solid #9C27B0; padding: 6px;'
                    elif val == "O":
                        df_style.iloc[r, c] = 'color: gray; text-align: center; border-bottom: 3px solid #9C27B0; padding: 6px;'
                else:  # This is a Token Row
                    if val != "":
                        df_style.iloc[r, c] = 'font-weight: bold; color: black; background-color: #f4f4f4; text-align: center; font-size: 14px; padding: 6px;'
        return df_style

    # Render the styled grid
    styled_grid = (grid_df.style
        .hide(axis="index")
        .hide(axis="columns")
        .apply(style_pred_grid, axis=None)
    )
    display(styled_grid)
    display(Markdown("<br>"))  # Adds a small space between different sentences

# ==========================================
# RUN PREDICTIONS
# ==========================================
display(Markdown("##Model Predictions (Manual Testing)"))

test_sentences = [
    "The patient with chronic kidney disease was treated with Lisinopril and showed improvement.",
    "Ibuprofen may worsen gastric ulcers in patients with peptic ulcer disease.",
    "She was diagnosed with type 2 diabetes and prescribed Metformin and insulin therapy.",
    "Long-term use of corticosteroids like Prednisone can lead to osteoporosis.",
    "The combination of Aspirin and Warfarin increases the risk of bleeding complications."
]

for s in test_sentences:
    predict_sentence_styled(s)

##Model Predictions (Manual Testing)

**Input:** *The patient with chronic kidney disease was treated with Lisinopril and showed improvement.*

The,patient,with,chronic,kidney,disease,was,treated,with,Lisinopril,and,showed
O,O,O,B-Disease,I-Disease,I-Disease,O,O,O,B-Chemical,O,O
improvement,.,,,,,,,,,,
O,O,,,,,,,,,,


<br>

**Input:** *Ibuprofen may worsen gastric ulcers in patients with peptic ulcer disease.*

Ibuprofen,may,worsen,gastric,ulcers,in,patients,with,peptic,ulcer,disease,.
B-Chemical,O,O,O,B-Disease,O,O,O,B-Disease,I-Disease,I-Disease,O


... (6 outputs left out)

<br>

**Input:** *The combination of Aspirin and Warfarin increases the risk of bleeding complications.*

The,combination,of,Aspirin,and,Warfarin,increases,the,risk,of,bleeding,complications
O,O,O,B-Chemical,O,B-Chemical,O,O,O,O,B-Disease,I-Disease
.,,,,,,,,,,,
O,,,,,,,,,,,


<br>

# Save Model


In [14]:
import torch
import json
import shutil
from google.colab import files

# 1. Create folder
save_path = "./bilstm_model"
import os
os.makedirs(save_path, exist_ok=True)

# 2. Save model weights
torch.save(model.state_dict(), f"{save_path}/bilstm_crf.pth")

# 3. Save vocab
with open(f"{save_path}/vocab.json", "w") as f:
    json.dump(word2idx, f)

# 4. Save labels
with open(f"{save_path}/tag2idx.json", "w") as f:
    json.dump(tag2idx, f)

# 5. Zip everything
shutil.make_archive("bilstm_model", 'zip', save_path)

# 6. Download
files.download("bilstm_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import torch
import pickle
import os
import shutil
from google.colab import files

# 1. Create folder
save_path = "bilstm_crf_export"
os.makedirs(save_path, exist_ok=True)

# 2. Save model weights
torch.save(model.state_dict(), f"{save_path}/model.pth")

# 3. Save mappings
with open(f"{save_path}/word2idx.pkl", "wb") as f:
    pickle.dump(word2idx, f)

with open(f"{save_path}/tag2idx.pkl", "wb") as f:
    pickle.dump(tag2idx, f)

with open(f"{save_path}/idx2tag.pkl", "wb") as f:
    pickle.dump(idx2tag, f)

# 4. Zip folder
shutil.make_archive("bilstm_crf_export", 'zip', save_path)

# 5. Download
files.download("bilstm_crf_export.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Save Resutls

In [16]:
from seqeval.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report
import json
import pandas as pd
import os

MODEL_NAME = "BiLSTM-CRF"

# =========================
# GLOBAL METRICS (TEST SET)
# =========================
precision = precision_score(true_labels, pred_labels)
recall = recall_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels)
accuracy = accuracy_score(true_labels, pred_labels)

# =========================
# PER-ENTITY METRICS
# =========================
report = classification_report(true_labels, pred_labels, output_dict=True)

# Safe extraction (avoids crash if key missing)
chemical = report.get("Chemical", {})
disease = report.get("Disease", {})

results = {
    "model": MODEL_NAME,

    # 🔹 Global
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "accuracy": accuracy,

    # 🔹 Chemical
    "Chemical_precision": chemical.get("precision", 0),
    "Chemical_recall": chemical.get("recall", 0),
    "Chemical_f1": chemical.get("f1-score", 0),

    # 🔹 Disease
    "Disease_precision": disease.get("precision", 0),
    "Disease_recall": disease.get("recall", 0),
    "Disease_f1": disease.get("f1-score", 0),
}

# =========================
# SAVE RESULTS
# =========================
os.makedirs("results", exist_ok=True)

json_path = f"results/{MODEL_NAME}.json"
csv_path = f"results/{MODEL_NAME}.csv"

with open(json_path, "w") as f:
    json.dump(results, f, indent=4)

pd.DataFrame([results]).to_csv(csv_path, index=False)

# =========================
# OUTPUT
# =========================
print("BiLSTM results saved!")
print("JSON:", json_path)
print("CSV:", csv_path)
print(results)

BiLSTM results saved!
JSON: results/BiLSTM-CRF.json
CSV: results/BiLSTM-CRF.csv
{'model': 'BiLSTM-CRF', 'precision': np.float64(0.6137761377613776), 'recall': np.float64(0.559730790802019), 'f1_score': np.float64(0.5855089469052508), 'accuracy': 0.9481686692652128, 'Chemical_precision': np.float64(0.5839267548321465), 'Chemical_recall': np.float64(0.6399108138238573), 'Chemical_f1': np.float64(0.6106382978723405), 'Disease_precision': np.float64(0.6594090202177294), 'Disease_recall': np.float64(0.4785553047404063), 'Disease_f1': np.float64(0.5546108567691301)}
